In [ ]:
# Data Cleaning
from datetime import datetime, timedelta
import json
import pandas as pd
import numpy as np
import re
import spacy

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from  matplotlib.ticker import PercentFormatter
import seaborn as sns

In [ ]:
def emotionGroup(
    emotion: str,
    valence: bool = False
) -> str:

    data = {
        'admiration': ['Affection', 'Positive'],
        'amusement': ['Happiness', 'Positive'],
        'anger': ['Anger', 'Negative'],
        'annoyance': ['Anger', 'Negative'],
        'approval': ['Satisfaction', 'Positive'],
        'caring': ['Affection', 'Positive'],
        'confusion': ['Fear', 'Negative'],
        'curiosity': ['Fear', 'Negative'],
        'desire': ['Fear', 'Negative'],
        'disappointment': ['Depression', 'Negative'],
        'disapproval': ['Anger', 'Negative'],
        'disgust': ['Contempt', 'Negative'],
        'embarrassment': ['Depression', 'Negative'],
        'excitement': ['Happiness', 'Positive'],
        'fear': ['Fear', 'Negative'],
        'gratitude': ['Satisfaction', 'Positive'],
        'grief': ['Depression', 'Negative'],
        'joy': ['Happiness', 'Positive'],
        'love': ['Affection', 'Positive'],
        'nervousness': ['Fear', 'Negative'],
        'optimism': ['Happiness', 'Positive'],
        'pride': ['Affection', 'Positive'],
        'realization': ['Satisfaction', 'Positive'],
        'relief': ['Happiness', 'Positive'],
        'remorse': ['Contempt', 'Negative'],
        'sadness': ['Depression', 'Negative'],
        'surprise': ['Happiness', 'Positive'],
        'neutral': ['Neutral', 'Neutral'],
        'error': ['Neutral', 'Neutral']
    }

    if emotion.lower() == 'error':
        return 'error'
    else:
        if valence:
            return data[emotion.lower()][1]
        else:
            return data[emotion.lower()][0]

In [ ]:
def plotLine(
    df,
    variable: str,
    period: list = []
):

    # Subset data
    if len(period) != 0:
        df_subset = df[(df['date'] >= datetime.strptime(period[0], '%Y-%m-%d').date())
                       & (df['date'] <= datetime.strptime(period[1], '%Y-%m-%d').date())]
    else:
        df_subset = df

    # Create plot
    # sns.set(rc={'figure.figsize':(11.7,3.27)})
    sns.set_style(style='white')
    plot = sns.lineplot(x='date', y=variable, data=df_subset)
    # Formatting
    # x-axis
    plot.set(xlabel=None)
    plt.xticks(rotation=45)
    # y-axis
    plot.set(ylabel='Total # of Comments')
    plot.yaxis.set_major_formatter(lambda y, p: f'{y/1000:.0f}k')
    # sns.reset_orig()
    # sns.set_style(style='white')

    return plot

In [ ]:
def plotStack(
    df,
    period: list = [],
    emotions: list = [],
    valence: bool = False,
    width: float = 1.0
    ):

    # Assign colours to emotions and valence
    mapEmotion = {
        'Affection': 'deepskyblue',
        'Happiness': 'green',
        'Satisfaction': 'turquoise',
        'Anger': 'firebrick',
        'Contempt': 'darkorange',
        'Depression': 'maroon',
        'Fear': 'gold',
        'Neutral': 'yellow'
    }
    mapValence = {
        'Positive': 'green',
        'Negative': 'firebrick'
    }
    if valence:
        mapper = mapValence
        sentiment = 'valence'
    else:
        mapper = mapEmotion
        sentiment = 'emotion'

    # Subset period
    if len(period) > 0:
        df_subset = df[(df['date'] >= datetime.strptime(period[0], '%Y-%m-%d').date()) & (df['date'] <= datetime.strptime(period[1], '%Y-%m-%d').date())]
    else:
        df_subset = df.copy(deep=True)

    # Unstack
    df_stack = df_subset[~df_subset.emotion.isin(['Neutral', 'error'])][['date', sentiment]]\
        .groupby('date')[sentiment]\
        .value_counts(normalize=False)\
        .unstack(sentiment)\
        .reset_index()

    df_stack100 = df_subset[~df_subset.emotion.isin(['Neutral', 'error'])][['date', sentiment]]\
        .groupby('date')[sentiment]\
        .value_counts(normalize=True)\
        .unstack(sentiment)\
        .reset_index()

    # Subset emotions
    if len(emotions) > 0:
        orderMap = [e for e in mapper if e != 'Neutral' and e in emotions]
        orderCol = ['date']+orderMap
        colourMap = [mapper[e] for e in mapper if e != 'Neutral' and e in emotions]
        df_stack = df_stack[orderCol]
    else:
        orderMap = [e for e in mapper if e != 'Neutral']
        orderCol = ['date']+orderMap
        colourMap = [mapper[e] for e in mapper if e != 'Neutral']

    # Create plot
    plot_stack = df_stack[orderCol].plot(x='date', kind='bar', stacked=True, color=colourMap, width=width)
    plot_stack.set(xlabel=None)
    plot_stack.legend_.set_title(None)
    plot_stack.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plot_stack.yaxis.set_major_formatter(lambda y, p: f'{y/1000:.0f}k')
    plt.xticks(rotation=45)

    plot_stack100 = df_stack100[orderCol].plot(x='date', kind='bar', stacked=True, color=colourMap, width=width)
    plot_stack100.set(xlabel=None)
    plot_stack100.legend_.set_title(None)
    plot_stack100.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plot_stack100.yaxis.set_major_formatter(lambda y, p: f'{y*100:.0f}'+'%')
    plt.xticks(rotation=45)
    plt.legend(loc='lower left')

    return df_stack, df_stack100

In [ ]:
def noOutlier(
    df,
    cols: list,
    sd: float = 3
    ):

    # Create copy
    output = df.copy(deep=True)

    # Drop rows with outliers
    for c in cols:
        UB = np.mean(output[c]) + (sd * np.std(output[c]))
        output = output[output[c] <= UB]

    return output

In [ ]:
def describeEvent(
    df,
    cols: list,
    periods: list = None,
    clustered: bool = False,
    func: list = ['count', 'mean', 'std', 'var', 'min', 'max']
    ):
    # Overall
    overall = df[cols].agg(func).reset_index()

    # Periods
    if periods:
        end = max(df['date'])
        # Create phase lists
        phases = [datetime.strptime(p, '%Y-%m-%d').date() for p in phases]
        phase_list = [[phases[0], phases[1]-timedelta(days=1)]]
        for i in range(1, len(phases)):
            if i < len(phases)-1:
                phase_list.append([phases[i], phases[i+1]-timedelta(days=1)])
            else:
                phase_list.append([phases[i], end])
        # Create labels
        def labeller(x):
            for p in phase_list:
                if x >= p[0] and x <= p[1]:
                    return p[0]
        # Label data
        df_labelled = df.copy(deep=True)
        df_labelled['phase'] = df_labelled['date'].apply(lambda x: labeller(x))
        # Aggregate
        phased = df[['phase']+cols].groupby('phase').agg(func).reset_index()

        return overall, phased

    # Clusters
    if clustered:
        agg_func = {c: ['mean', 'std', 'var', 'min', 'max'] for c in cols}
        agg_func['date'] = ['min', 'max']
        return df.groupby('phase').agg(agg_func).reset_index()

    return overall